# Разбиение выборки, метрики и базовые модели

Блокнот связан с четвертой главой пособия и предназначен для отработки корректной схемы оценки качества до перехода к прикладным кейсам.

## Цель и ожидаемые результаты

После выполнения блокнота читатель должен:

- различать обучающую и тестовую выборки;
- понимать роль baseline-модели;
- уметь выбирать метрики под тип задачи;
- видеть, почему качество нельзя оценивать на тех же данных, на которых модель обучалась.

## Постановка задачи

Необходимо построить простую baseline-модель для регрессионной постановки, рассчитать базовые метрики и интерпретировать полученные значения.

## Используемые библиотеки

| Библиотека | Назначение |
| --- | --- |
| pandas / numpy | работа с признаками и целевой переменной |
| scikit-learn | разбиение выборки, baseline и метрики |
| src.display_utils | структурированный вывод метрик и выводов |

## Источник и описание данных

В блокноте используется небольшой учебный набор с признаками температуры, влажности и давления, а также целевой переменной `power_output_mw`, имитирующей регрессионную постановку для энергетической системы.

## Перечень признаков

| Признак | Тип | Смысл |
| --- | --- | --- |
| temperature_c | float | температура внешней среды |
| humidity_pct | float | относительная влажность |
| pressure_kpa | float | атмосферное давление |
| power_output_mw | float | выходная мощность |

## Этапы анализа

### 1. Формирование учебного набора
Создается компактная таблица наблюдений.

### 2. Разбиение выборки
Данные разделяются на обучающую и тестовую части.

### 3. Построение baseline и простой модели
Сравниваются наивный подход и линейная регрессия.

### 4. Интерпретация метрик
Оценивается практический смысл ошибки прогноза.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from src.display_utils import display_callout, display_metric_table

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

df = pd.DataFrame(
    {
        "temperature_c": [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24],
        "humidity_pct": [78, 75, 72, 70, 68, 65, 63, 60, 58, 56, 54, 52],
        "pressure_kpa": [100.2, 100.3, 100.1, 100.0, 99.9, 99.8, 99.7, 99.7, 99.6, 99.5, 99.4, 99.4],
        "power_output_mw": [465, 462, 458, 454, 451, 447, 443, 439, 435, 432, 428, 424],
    }
)
df

## Интерпретация

Учебный набор демонстрирует задачу регрессии с небольшим числом признаков. Даже на таком примере можно корректно показать различие между baseline и более содержательной моделью.

In [ ]:
X = df[["temperature_c", "humidity_pct", "pressure_kpa"]]
y = df["power_output_mw"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

split_summary = pd.DataFrame(
    {
        "subset": ["train", "test"],
        "rows": [len(X_train), len(X_test)],
    }
)
split_summary

## Интерпретация

Разбиение фиксирует независимую тестовую часть, на которой будет оцениваться качество. Без такого разделения нельзя получить содержательную оценку модели.

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

model = LinearRegression()
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

baseline_metrics = {
    "MAE": mean_absolute_error(y_test, base_pred),
    "RMSE": mean_squared_error(y_test, base_pred) ** 0.5,
    "R2": r2_score(y_test, base_pred),
}

model_metrics = {
    "MAE": mean_absolute_error(y_test, model_pred),
    "RMSE": mean_squared_error(y_test, model_pred) ** 0.5,
    "R2": r2_score(y_test, model_pred),
}

metrics_df = pd.DataFrame([baseline_metrics, model_metrics], index=["baseline", "linear_regression"])
metrics_df

## Интерпретация

Baseline-модель задает минимальный ориентир для сравнения. Если более сложная модель не улучшает эту точку отсчета, то дальнейшее усложнение не имеет методического смысла.

In [ ]:
display_metric_table(
    {
        "baseline_mae": baseline_metrics["MAE"],
        "model_mae": model_metrics["MAE"],
        "baseline_rmse": baseline_metrics["RMSE"],
        "model_rmse": model_metrics["RMSE"],
    },
    decimals=3,
)

display_callout(
    "Итог интерпретации",
    "Метрики должны рассматриваться не изолированно, а в связи с допустимой инженерной ошибкой и схемой формирования выборки.",
    tone="success",
)

## Итоговые выводы

- Разбиение выборки является обязательной частью вычислительного эксперимента.
- Baseline-модель задает минимальный уровень, который должна превзойти содержательная модель.
- Метрики следует выбирать исходя из задачи и физического смысла ошибки, а не только из привычки использовать один показатель.

## Вопросы и задания для самостоятельной работы

1. Добавьте валидационную выборку и сравните структуру эксперимента.
2. Почему `R2` не должен использоваться как единственная метрика качества?
3. Как изменилась бы схема оценки для временного ряда?

## Источники

1. [Глава 4 пособия](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/chapters/04-splitting-metrics-and-baselines.md)
2. [Outline пособия](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/handbook-outline.md)